In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

In [5]:
results_file = '/Users/tomasandrade/Documents/BSC/ICHOIR/applio/Applio_LS/asv_experiments/scores_multiscale.csv'
df = pd.read_csv(results_file)

In [8]:
horizons = [1, 5, 10, 25, 50]
score_cols = [f'score_h{h}' for h in horizons]

# Ratio score
df['score_ratio'] = df['score_h50'] / df['score_h1'].clip(lower=1e-6)

# Slope score
log_h = np.log(horizons)
df['score_slope'] = df[score_cols].apply(
    lambda row: np.polyfit(log_h, row.values, 1)[0], axis=1
)

y_true = (df['key'] == 'spoof').astype(int)
for col in ['score', 'score_ratio', 'score_slope', 'score_h1', 'score_h50']:
    auc = roc_auc_score(y_true, df[col])
    print(f"{col:20s}  AUC {auc:.4f}  (flipped: {1-auc:.4f})")

score                 AUC 0.4871  (flipped: 0.5129)
score_ratio           AUC 0.5536  (flipped: 0.4464)
score_slope           AUC 0.5482  (flipped: 0.4518)
score_h1              AUC 0.3901  (flipped: 0.6099)
score_h50             AUC 0.5069  (flipped: 0.4931)


In [ ]:
import pandas as pd

In [2]:
df = pd.read_csv('~/Desktop/silence_diagnostics/flagged_silences.csv')

# Count how many flags each file has
flag_cols = ['flag_long_internal', 'flag_high_fraction', 
             'flag_short_speech', 'flag_many_segments']
df['n_flags'] = df[flag_cols].sum(axis=1)

# Worst offenders — multiple flags AND long internal silence
priority = df[df['n_flags'] >= 2].sort_values(
    'max_internal_sil_s', ascending=False
)
print(f"Files with 2+ flags: {len(priority)}")
print(priority[['name', 'duration_s', 'max_internal_sil_s', 
                'silence_fraction', 'n_flags']].head(30))

Files with 2+ flags: 422
            name  duration_s  max_internal_sil_s  silence_fraction  n_flags
0   LA_T_4982302       6.400                3.98             0.656        2
1   LA_T_9215136       6.440                3.92             0.630        2
2   LA_T_1129020       8.500                3.40             0.786        2
3   LA_T_9690282       4.620                3.26             0.753        2
4   LA_T_3448136       4.580                3.22             0.729        2
5   LA_T_7521161       8.880                2.80             0.869        2
6   LA_T_6960204       4.900                1.98             0.547        2
7   LA_T_8503056       4.400                1.94             0.700        2
8   LA_T_5480079       2.940                1.94             0.741        2
9   LA_T_6010560       4.460                1.68             0.628        2
10  LA_T_5706600       4.420                1.64             0.665        2
11  LA_T_8719032       4.420                1.64             0.

In [6]:
cm_file = '/Users/tomasandrade/Documents/BSC/anti-spoof/dataset/ASVspoof2019/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
protocol = pd.read_csv(
    cm_file,
    sep=' ', header=None,
    names=['speaker_id', 'file_name', 'placeholder', 'system_id', 'key']
)

# Merge on filename (flagged 'name' matches protocol 'file_name')
flagged = df.merge(
    protocol[['file_name', 'key', 'system_id']],
    left_on='name', right_on='file_name',
    how='left'
)

In [7]:
print(flagged['key'].value_counts())
print(flagged['key'].value_counts(normalize=True))

key
spoof       2312
bonafide     100
Name: count, dtype: int64
key
spoof       0.958541
bonafide    0.041459
Name: proportion, dtype: float64


In [8]:
print(flagged[flagged['key']=='spoof']['system_id'].value_counts())
print(flagged[flagged['key']=='spoof']['system_id'].value_counts(normalize=True))

system_id
A04    772
A01    519
A03    333
A02    331
A05    214
A06    143
Name: count, dtype: int64
system_id
A04    0.333910
A01    0.224481
A03    0.144031
A02    0.143166
A05    0.092561
A06    0.061851
Name: proportion, dtype: float64
